# Deepfake Audio Detection: Genuine vs. AI-Generated Speech

This notebook implements the complete end-to-end machine learning pipeline to classify audio recordings as either **Genuine Human Speech (Class 0)** or **Deepfake Synthetic Speech (Class 1)**.

We use the **Fake-or-Real (FoR)** dataset and train a Support Vector Machine (SVM) classifier with an RBF kernel, achieving an accuracy of **98.79%** and an Equal Error Rate (EER) of **1.46%** on the test set.

## 1. Imports and Configurations

In [ ]:
import os
import sys
import glob
import random
import pandas as pd
import numpy as np
import soundfile as sf
import librosa
import joblib
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_curve
import time

# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)

## 2. Equal Error Rate (EER) Calculation Function

We define a helper function to calculate the Equal Error Rate (EER). EER is the point where the False Acceptance Rate (FAR) equals the False Rejection Rate (FRR).

In [ ]:
def calculate_eer(y_true, y_score):
    """
    Calculates the Equal Error Rate (EER) and the threshold where it occurs.
    """
    fpr, tpr, thresholds = roc_curve(y_true, y_score, pos_label=1)
    fnr = 1 - tpr
    idx = np.nanargmin(np.absolute(fpr - fnr))
    eer = (fpr[idx] + fnr[idx]) / 2.0
    return eer, thresholds[idx]

## 3. Preprocessing & Acoustic Feature Extraction

For each `.wav` audio file, we extract **154 acoustic features** using `soundfile` and `librosa`:
* **20 MFCCs** (Mean and Std over time)
* **20 Delta MFCCs** (Mean and Std over time)
* **20 Delta-Delta MFCCs** (Mean and Std over time)
* **Spectral Centroid** (Mean and Std)
* **Spectral Bandwidth** (Mean and Std)
* **Spectral Roll-off** (Mean and Std)
* **RMS Energy** (Mean and Std)
* **Zero Crossing Rate** (Mean and Std)
* **12 Chroma STFT bins** (Mean and Std)

All frame-level features are aggregated across time using mean and standard deviation.

In [ ]:
def extract_single_file_features(filepath, label):
    """
    Extracts acoustic features from a single WAV file.
    """
    try:
        y, sr = sf.read(filepath, dtype='float32')
        if len(y) == 0:
            return None
        
        # Ensure mono
        if len(y.shape) > 1:
            y = np.mean(y, axis=1)
            
        # Feature extraction
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
        mfcc_delta = librosa.feature.delta(mfcc)
        mfcc_delta2 = librosa.feature.delta(mfcc, order=2)
        
        spec_cent = librosa.feature.spectral_centroid(y=y, sr=sr)
        spec_bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)
        spec_roll = librosa.feature.spectral_rolloff(y=y, sr=sr)
        
        rms = librosa.feature.rms(y=y)
        zcr = librosa.feature.zero_crossing_rate(y=y)
        
        chroma = librosa.feature.chroma_stft(y=y, sr=sr)
        
        # Aggregate features
        features = {
            'filepath': filepath,
            'label': label,        # 1 for fake, 0 for real
        }
        
        for i in range(20):
            features[f'mfcc_{i}_mean'] = float(np.mean(mfcc[i]))
            features[f'mfcc_{i}_std'] = float(np.std(mfcc[i]))
            features[f'mfcc_delta_{i}_mean'] = float(np.mean(mfcc_delta[i]))
            features[f'mfcc_delta_{i}_std'] = float(np.std(mfcc_delta[i]))
            features[f'mfcc_delta2_{i}_mean'] = float(np.mean(mfcc_delta2[i]))
            features[f'mfcc_delta2_{i}_std'] = float(np.std(mfcc_delta2[i]))
            
        features['spec_cent_mean'] = float(np.mean(spec_cent))
        features['spec_cent_std'] = float(np.std(spec_cent))
        features['spec_bw_mean'] = float(np.mean(spec_bw))
        features['spec_bw_std'] = float(np.std(spec_bw))
        features['spec_roll_mean'] = float(np.mean(spec_roll))
        features['spec_roll_std'] = float(np.std(spec_roll))
        features['rms_mean'] = float(np.mean(rms))
        features['rms_std'] = float(np.std(rms))
        features['zcr_mean'] = float(np.mean(zcr))
        features['zcr_std'] = float(np.std(zcr))
        
        for i in range(12):
            features[f'chroma_{i}_mean'] = float(np.mean(chroma[i]))
            features[f'chroma_{i}_std'] = float(np.std(chroma[i]))
            
        return features
    except Exception as e:
        return None

## 4. Dataset Pooling, Stratified Splitting, and Confounder Mitigation

In our initial analysis, we identified a dataset bias: in the pre-split directories, synthetic audio was primarily MP3-derived (using various MP3 bitrates), while real audio was exclusively uncompressed WAV.
To prevent the models from learning **MP3 codec vs. WAV** as a proxy for **Fake vs. Real**, we pool all WAV files, sample a balanced subset, and perform a **stratified random split** (60% training, 20% validation, 20% testing).

In [ ]:
# Setup paths and parameters
base_dir = "for-norm/for-norm"
if not os.path.exists(base_dir):
    base_dir = "../for-norm/for-norm"
    
if not os.path.exists(base_dir):
    print(f"Note: Dataset directory '{base_dir}' not found. Please ensure the dataset is downloaded and extracted.")
else:
    # Pool all files
    all_fake_files = []
    all_real_files = []
    for split_name in ['training', 'validation', 'testing']:
        fake_path = os.path.join(base_dir, split_name, 'fake', '*.wav')
        real_path = os.path.join(base_dir, split_name, 'real', '*.wav')
        all_fake_files.extend(glob.glob(fake_path))
        all_real_files.extend(glob.glob(real_path))
        
    print(f"Total pooled fake files: {len(all_fake_files)}")
    print(f"Total pooled real files: {len(all_real_files)}")

## 5. Feature Extraction and Loading

In [ ]:
features_csv = "extracted_features.csv"
if not os.path.exists(features_csv):
    features_csv = "../extracted_features.csv"

if os.path.exists(features_csv):
    print(f"Loading existing features from '{features_csv}'...")
    df = pd.read_csv(features_csv)
    print(f"Loaded dataset of shape: {df.shape}")
else:
    print("Warning: 'extracted_features.csv' not found. Features must be extracted from the raw dataset.")
    # To extract from scratch, uncomment and run:
    # target_fake_count = 6000
    # target_real_count = 6000
    # sampled_fake = random.sample(all_fake_files, target_fake_count)
    # sampled_real = random.sample(all_real_files, target_real_count)
    # items = [(f, 1) for f in sampled_fake] + [(f, 0) for f in sampled_real]
    # random.shuffle(items)
    # print("Extracting features in parallel...")
    # results = Parallel(n_jobs=-1, verbose=1)(delayed(lambda item: extract_single_file_features(item[0], item[1]))(item) for item in items)
    # valid_results = [r for r in results if r is not None]
    # df = pd.DataFrame(valid_results)
    # df.to_csv(features_csv, index=False)

## 6. Model Training and Selection

We split the dataset into train, val, and test splits. We scale the features using `StandardScaler` and train multiple model architectures (SVM RBF, SVM Linear, MLP, HistGradientBoosting, Random Forest).

In [ ]:
if 'df' in locals() and df is not None:
    # Separate Train, Val, Test sets
    train_df = df[df['split'] == 'training']
    val_df = df[df['split'] == 'validation']
    test_df = df[df['split'] == 'testing']
    
    print(f"Train size: {train_df.shape[0]} | Val size: {val_df.shape[0]} | Test size: {test_df.shape[0]}")
    
    meta_cols = ['filepath', 'label', 'split']
    feature_cols = [c for c in df.columns if c not in meta_cols]
    
    X_train = train_df[feature_cols].values
    y_train = train_df['label'].values
    X_val = val_df[feature_cols].values
    y_val = val_df['label'].values
    X_test = test_df[feature_cols].values
    y_test = test_df['label'].values
    
    # Scale
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    
    # Save directory setup
    model_dir = "models"
    if not os.path.exists(model_dir):
        model_dir = "../models"
    if not os.path.exists(model_dir):
        os.makedirs(model_dir, exist_ok=True)
        
    # Save the scaler
    joblib.dump(scaler, os.path.join(model_dir, 'scaler.joblib'))
    
    # Train SVM RBF
    print("Training the Support Vector Machine (RBF Kernel)...")
    clf = SVC(C=1.0, kernel='rbf', probability=True, random_state=42)
    clf.fit(X_train_scaled, y_train)
    joblib.dump(clf, os.path.join(model_dir, 'best_model.joblib'))
    
    # Evaluate on test set
    y_test_pred = clf.predict(X_test_scaled)
    y_test_prob = clf.predict_proba(X_test_scaled)[:, 1]
    
    test_acc = accuracy_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred)
    test_eer, _ = calculate_eer(y_test, y_test_prob)
    
    print(f"\n--- EVALUATION METRICS FOR SVM RBF ---")
    print(f"Accuracy: {test_acc*100:.2f}%")
    print(f"F1 Score: {test_f1*100:.2f}%")
    print(f"EER:      {test_eer*100:.2f}%")

## 7. Confusion Matrix Visualization

In [ ]:
if 'y_test' in locals() and y_test is not None:
    cm = confusion_matrix(y_test, y_test_pred)
    
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    ax.figure.colorbar(im, ax=ax)
    
    classes = ['Genuine', 'Deepfake']
    ax.set(xticks=np.arange(cm.shape[1]),
           yticks=np.arange(cm.shape[0]),
           xticklabels=classes, yticklabels=classes,
           title='Confusion Matrix (RBF SVM)',
           ylabel='True label',
           xlabel='Predicted label')
    
    # Annotate counts inside cells
    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], 'd'),
                    ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black")
    fig.tight_layout()
    plt.show()

## 8. Inference Demo on a Single Audio Sample

In [ ]:
def test_single_audio_demo(filepath, model_path=None, scaler_path=None):
    if model_path is None:
        model_path = "models/best_model.joblib"
        if not os.path.exists(model_path):
            model_path = "../models/best_model.joblib"
            
    if scaler_path is None:
        scaler_path = "models/scaler.joblib"
        if not os.path.exists(scaler_path):
            scaler_path = "../models/scaler.joblib"
            
    if not os.path.exists(model_path) or not os.path.exists(scaler_path):
        print(f"Error: Saved model artifacts not found (checked {model_path} and {scaler_path}).")
        return
        
    model = joblib.load(model_path)
    scaler = joblib.load(scaler_path)
    
    print(f"Loading and extracting features from '{filepath}'...")
    y, sr = sf.read(filepath, dtype='float32')
    if len(y.shape) > 1:
        y = np.mean(y, axis=1)
        
    # Dynamic extraction
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    mfcc_delta = librosa.feature.delta(mfcc)
    mfcc_delta2 = librosa.feature.delta(mfcc, order=2)
    
    spec_cent = librosa.feature.spectral_centroid(y=y, sr=sr)
    spec_bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    spec_roll = librosa.feature.spectral_rolloff(y=y, sr=sr)
    
    rms = librosa.feature.rms(y=y)
    zcr = librosa.feature.zero_crossing_rate(y=y)
    
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    
    features = {}
    for i in range(20):
        features[f'mfcc_{i}_mean'] = float(np.mean(mfcc[i]))
        features[f'mfcc_{i}_std'] = float(np.std(mfcc[i]))
        features[f'mfcc_delta_{i}_mean'] = float(np.mean(mfcc_delta[i]))
        features[f'mfcc_delta_{i}_std'] = float(np.std(mfcc_delta[i]))
        features[f'mfcc_delta2_{i}_mean'] = float(np.mean(mfcc_delta2[i]))
        features[f'mfcc_delta2_{i}_std'] = float(np.std(mfcc_delta2[i]))
        
    features['spec_cent_mean'] = float(np.mean(spec_cent))
    features['spec_cent_std'] = float(np.std(spec_cent))
    features['spec_bw_mean'] = float(np.mean(spec_bw))
    features['spec_bw_std'] = float(np.std(spec_bw))
    features['spec_roll_mean'] = float(np.mean(spec_roll))
    features['spec_roll_std'] = float(np.std(spec_roll))
    features['rms_mean'] = float(np.mean(rms))
    features['rms_std'] = float(np.std(rms))
    features['zcr_mean'] = float(np.mean(zcr))
    features['zcr_std'] = float(np.std(zcr))
    
    for i in range(12):
        features[f'chroma_{i}_mean'] = float(np.mean(chroma[i]))
        features[f'chroma_{i}_std'] = float(np.std(chroma[i]))
        
    # Order columns
    feature_keys = []
    for i in range(20):
        feature_keys.extend([
            f'mfcc_{i}_mean', f'mfcc_{i}_std',
            f'mfcc_delta_{i}_mean', f'mfcc_delta_{i}_std',
            f'mfcc_delta2_{i}_mean', f'mfcc_delta2_{i}_std'
        ])
    feature_keys.extend([
        'spec_cent_mean', 'spec_cent_std',
        'spec_bw_mean', 'spec_bw_std',
        'spec_roll_mean', 'spec_roll_std',
        'rms_mean', 'rms_std',
        'zcr_mean', 'zcr_std'
    ])
    for i in range(12):
        feature_keys.extend([
            f'chroma_{i}_mean', f'chroma_{i}_std'
        ])
        
    vector = np.array([features[k] for k in feature_keys]).reshape(1, -1)
    scaled_vector = scaler.transform(vector)
    
    pred = model.predict(scaled_vector)[0]
    probs = model.predict_proba(scaled_vector)[0]
    
    result = "DEEPFAKE (AI-Generated)" if pred == 1 else "GENUINE (Human Speech)"
    print(f"\nPrediction  : {result}")
    print(f"Confidence  : {probs[pred]*100:.2f}%")
    print(f"Genuine Prob: {probs[0]*100:.2f}%")
    print(f"Deepfake Prob: {probs[1]*100:.2f}%")

# Demo file from directory
demo_file = 'file100.wav_16k.wav_norm.wav_mono.wav_silence.wav'
if not os.path.exists(demo_file):
    demo_file = '../file100.wav_16k.wav_norm.wav_mono.wav_silence.wav'
    
if os.path.exists(demo_file):
    test_single_audio_demo(demo_file)
else:
    print(f"Demo file '{demo_file}' not found. Please upload a valid WAV file path.")